# 08 — Brain states × mapas de referência

Interpreta uma definição de cluster (um conjunto de brain states) em termos neuroanatômicos: a ativação média de cada estado no córtex e no subcórtex, e a correlação com mapas **moleculares** (PET, neuromaps), **comportamentais** (taxonomia BrainMap × Brainnetome) e **canônicos** (Yeo 7 e as nossas 14 redes).

Duas etapas, dois scripts nesta pasta (`notebooks/`):

1. `state_correlations.compute(...)` → **tabelas** (`state_means.csv`, `correlations.csv`, `provenance.json`) em `results/<estados>/<atlas>/`
2. `state_figures.draw(...)` → **figuras** a partir dessas tabelas (visão geral + um card por estado)

Mapas de referência: rode uma vez `notebooks/fetch_reference_maps.sh`. Detalhes e limitações: `notebooks/state_maps.md`.

In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "notebooks"))   # refmaps, state_correlations, state_figures
import state_correlations as sc
import state_figures as sf

## Parâmetros

**Definição de cluster** — `STATES`:
- `"simulate"`: seis estados simulados com resposta conhecida (para testar);
- uma coluna de estados do stage 4b, ex. `"HMM2_pca14_10"` (precisa de `OUTPUT_ROOT` e `WINDOW_S`);
- ou o caminho de um CSV escrito por `fmri-decomp state-means --csv`.

**Atlas** — `ATLASES`: onde os estados foram ajustados (`harvardoxford`, `networks_nodes`, `networks`, `yeo7`). Uma coluna de estados existe num atlas só; liste os atlas onde ela existe.

**O que mostrar** — `FAMILIES` (painéis não comportamentais) e `TAXONOMY`: `"brainmap"` (BrainMap, como o Brainnetome usa) ou `"beam2021"` (6 domínios data-driven de Beam et al. 2021: Memory, Reward, Cognition, Vision, Manipulation, Language). `BN_LEVELS`: profundidade — `"domain"` (os domínios), `"subcategory"` (BrainMap: ex. Vision · Motion; Beam: os termos de cada domínio), ou os dois.

**Estatística** — `N_NULLS` substitutos por mapa (`NULL="moran"`, segundos; `"burt"`, brainsmash, horas). Atlas com < 30 parcelas (`networks`, `yeo7`) ficam sem p-valor.

**Figura** — uma linha por estado: o cérebro em 2 × 3 (lateral E · lateral D · axial / medial E · medial D · coronal) e, ao lado, um gráfico de barras por família em `PANELS` (as `TOP` maiores |r|; categorias "Other" do BrainMap ficam de fora). As barras são a média de r entre sujeitos **± desvio padrão entre sujeitos** quando os estados vêm de uma coluna de estados ou da simulação (cada sujeito tem o seu mapa de estado); com um CSV de `state-means` só existe o mapa do grupo e as barras saem sem erro.

In [ ]:
STATES      = "simulate"            # "simulate" | "HMM2_pca14_10" | "path/to/state_means.csv"
WINDOW_S    = None                  # obrigatório p/ coluna de estados: -1 = per-TR (activation)
COHORTS     = "train"               # "train" = só as coortes de treino do decompose | lista | None = todas
OUTPUT_ROOT = None                  # None = $FMRIDECOMP_OUTPUTS ou output_root de config/camcan_movie.yaml

ATLASES     = ["harvardoxford"]     # + "networks_nodes", "networks", "yeo7"
FAMILIES    = ["molecular", "yeo7", "networks14"]
BN_LEVELS   = ["domain", "subcategory"]     # profundidade da taxonomia comportamental
BEHAVIOURAL = "neurosynth"          # perfis calculados do Neurosynth | CSV BrainMap real | None = simulado
TAXONOMY    = "brainmap"            # "brainmap" (Fox 2005, via Brainnetome) | "beam2021" (Beam et al. 2021)

PANELS      = ["yeo7", "behavioural"]   # gráficos ao lado de cada cérebro, nessa ordem
TOP         = 7                         # barras por gráfico (Yeo 7 mostra sempre as 7)
POSITIVE_ONLY = ["behavioural"]          # gráficos só com r positivo; [] = positivos e negativos

N_NULLS     = 10_000
NULL        = "moran"
RESULTS     = REPO / "notebooks" / "state_maps_results"

## 1. Tabelas de correlação

In [ ]:
dirs = sc.compute(
    STATES, ATLASES,
    families=FAMILIES + [sc.BN_LEVELS[b] for b in BN_LEVELS],
    out=RESULTS, output_root=OUTPUT_ROOT, window_s=WINDOW_S, cohorts=COHORTS,
    behavioural_csv=BEHAVIOURAL, n_nulls=N_NULLS, null=NULL, taxonomy=TAXONOMY,
)
run_dir = next(iter(dirs.values())).parent
run_dir

In [ ]:
import pandas as pd
cor = pd.concat({a: pd.read_csv(d / "correlations.csv") for a, d in dirs.items()}, names=["atlas"])
cor.sort_values("p_null").groupby(["atlas", "family"]).head(3)

## 2. Figuras

In [ ]:
paths = sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS,
                panels=PANELS, top=TOP, positive_only=POSITIVE_ONLY)
for p in paths:
    display(Image(filename=str(p)))

## 3. Matriz entre mapas de referência

Correlação espacial, entre as parcelas do atlas escolhido, entre cada mapa de uma família (linhas) e cada mapa de outra (colunas) — por padrão as 7 redes de Yeo × as 38 subcategorias do BrainMap (perfis de ativação do Neurosynth sobre o Brainnetome; sem as "Other"). Não depende dos estados. Asteriscos: `*` q<.05, `**` q<.01, contra mapas substitutos que preservam a autocorrelação espacial, FDR sobre a matriz inteira. Famílias: `molecular`, `behavioural`, `behavioural_domain`, `yeo7`, `networks14`.

In [ ]:
MATRIX_ATLAS = "harvardoxford"     # "harvardoxford" | "networks_nodes" | "networks" | "yeo7"
MATRIX_ROWS  = "yeo7"
MATRIX_COLS  = "behavioural"

png = sf.reference_matrix(MATRIX_ATLAS, MATRIX_ROWS, MATRIX_COLS, taxonomy=TAXONOMY,
                          behavioural_csv=BEHAVIOURAL, n_nulls=N_NULLS, null=NULL)
display(Image(filename=str(png)))

## 4. Outras visualizações (opcional)

Descomente para gerar. Usam **todas as famílias** de `FAMILIES` + `BN_LEVELS` (moleculares, comportamentais, Yeo 7, nossas 14 redes), não só `PANELS`.

In [ ]:
# Matriz: uma linha por estado (cérebro) e uma faixa de calor por família, com todas as famílias
# sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS, overview=True)
# for atlas in ATLASES:
#     display(Image(filename=str(dirs[atlas] / "state_maps_overview.png")))

In [ ]:
# Cards: um por estado, com barras de todas as famílias
# sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS, cards=True)
# for atlas in ATLASES:
#     for card in sorted(dirs[atlas].glob("state_card_S*.png"),
#                        key=lambda p: int(p.stem.rsplit("S", 1)[1])):
#         print(atlas, card.stem)
#         display(Image(filename=str(card)))